# Simple Linear Regression — Custom Class Implementation from Scratch

In this notebook we **build the OLS algorithm by hand**, inside a Python class called `MyLR` (My Linear Regression).

This is the best way to truly understand what happens inside `sklearn.linear_model.LinearRegression` — because we replicate it step-by-step ourselves.

---

## The Math We Are Implementing

Simple Linear Regression fits a straight line through data:

$$\hat{y} = m \cdot x + b$$

Where:
- $m$ = **slope** — how much $y$ changes per unit change in $x$
- $b$ = **intercept** — the value of $y$ when $x = 0$

The Ordinary Least Squares (OLS) formulas for the optimal $m$ and $b$ are:

$$m = \frac{\sum_{i=1}^{N}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{N}(x_i - \bar{x})^2}$$

$$b = \bar{y} - m \cdot \bar{x}$$

The `MyLR` class below computes exactly these two formulas using a simple `for` loop over the training data.

---

In [74]:
class MyLR:

    def __init__(self):
        self.m = None
        self.b = None

    def fit (self , x_train, y_train):
        numerator = 0
        denominator = 0
        for i in range(x_train.shape[0]):

          numerator = numerator + ((x_train[i] - x_train.mean())*(y_train[i] - y_train.mean()))
          denominator = denominator + ((x_train[i] - x_train.mean())**2)

        self.m = numerator/denominator      
        self.b = y_train.mean() - (self.m * x_train.mean())

        print("m is ", self.m)
        print("b is ", self.b)

    def predict (self, x_test):
        return self.m * x_test + self.b

---

### Explanation: `MyLR` Class — Line by Line

This is the heart of the notebook. Every method is explained in detail below.

---

#### `__init__(self)`

```python
self.m = None
self.b = None
```

The constructor initializes the slope (`m`) and intercept (`b`) to `None`.  
At this point no training has happened — these will be filled in by `.fit()`.

---

#### `fit(self, x_train, y_train)`

This method implements the **OLS closed-form solution** using a manual loop:

```python
numerator = 0
denominator = 0
```
Two accumulators are set to zero before the loop starts.

```python
for i in range(x_train.shape[0]):
```
Iterates over every training sample. `x_train.shape[0]` is the number of rows (students), i.e., 160.

```python
numerator += (x_train[i] - x_train.mean()) * (y_train[i] - y_train.mean())
```
Accumulates the **covariance** term:
$$\text{numerator} = \sum_{i=1}^{N}(x_i - \bar{x})(y_i - \bar{y})$$
- `x_train[i] - x_train.mean()` → how far this student's CGPA is from the average CGPA.
- `y_train[i] - y_train.mean()` → how far this student's salary is from the average salary.
- Multiplying these together measures whether both values deviate in the **same direction** (positive product) or **opposite directions** (negative product).

```python
denominator += (x_train[i] - x_train.mean()) ** 2
```
Accumulates the **variance** term:
$$\text{denominator} = \sum_{i=1}^{N}(x_i - \bar{x})^2$$
- Always positive (squared). Measures total spread of CGPA values around the mean.

```python
self.m = numerator / denominator
```
Computes the optimal slope:
$$m = \frac{\text{Covariance}(x, y)}{\text{Variance}(x)}$$

```python
self.b = y_train.mean() - (self.m * x_train.mean())
```
Computes the optimal intercept:
$$b = \bar{y} - m \cdot \bar{x}$$
This formula **guarantees** the regression line passes through the centroid $(\bar{x}, \bar{y})$ of the training data.

```python
print("m is ", self.m)
print("b is ", self.b)
```
Prints the learned parameters so we can inspect them immediately after training.

---

#### `predict(self, x_test)`

```python
return self.m * x_test + self.b
```
Applies the learned line equation to any new input $x$:
$$\hat{y} = m \cdot x + b$$

This works for a **single value** (`x_test[0]`) or a **NumPy array** of values (vectorized via NumPy broadcasting).

---

> **Key Insight:** This `fit()` loop and `sklearn`'s `LinearRegression().fit()` produce **identical** $m$ and $b$ values. The only difference is that `sklearn` uses optimized matrix algebra internally instead of a Python `for` loop — same math, faster execution.

---

In [75]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---

### Explanation: Library Imports

| Library | Alias | Purpose |
|---|---|---|
| `numpy` | `np` | N-dimensional arrays, math operations — the backbone of `MyLR` |
| `pandas` | `pd` | Loading and inspecting the CSV dataset |
| `matplotlib.pyplot` | `plt` | (Available for future visualization if needed) |

---

In [76]:
df = pd.read_csv("data/regression/placement.csv")
df.head()

,cgpa,package
0,6.89,3.26
1,5.12,1.98
2,7.82,3.25
3,7.42,3.67
4,6.94,3.57


---

### Explanation: Loading the Dataset

```python
df = pd.read_csv("data/regression/placement.csv")
df.head()
```

Loads the `placement.csv` file into a Pandas DataFrame and displays the first 5 rows.

| Column | ML Role | Meaning |
|---|---|---|
| `cgpa` | Input feature $x$ | Student's college GPA (scale ~4–10) |
| `package` | Target $y$ | Salary package offered, in LPA (Lakhs Per Annum) |

The dataset has **200 students**. Goal: predict salary package from CGPA.

---

In [77]:
x = df.iloc[:, 0].values #gives numpy array of the first column
y = df.iloc[:, 1].values #gives numpy array of the second column

---

### Explanation: Extracting Feature and Target Arrays

```python
x = df.iloc[:, 0].values   # CGPA  — 1D NumPy array
y = df.iloc[:, 1].values   # Package — 1D NumPy array
```

**`.values`** at the end converts the Pandas Series into a **plain NumPy array**.

This is important here because `MyLR.fit()` uses:
- `x_train[i]` — integer indexing (works on NumPy arrays, not always on Pandas Series)
- `x_train.mean()` — NumPy's mean method
- `x_train.shape[0]` — NumPy shape attribute

> **Difference from the sklearn notebook:** In the Scikit-Learn version we kept `X` as a 2D DataFrame `(200, 1)` because `sklearn` requires 2D input. Here, our custom `MyLR` class is written to accept **1D arrays**, so we extract flat arrays with `.values`.

After this cell:
- `x` has shape `(200,)` — a flat array of 200 CGPA values
- `y` has shape `(200,)` — a flat array of 200 salary values

---

In [78]:
x

array([6.89, 5.12, 7.82, 7.42, 6.94, 7.89, 6.73, 6.75, 6.09, 8.31, 5.32,
       6.61, 8.94, 6.93, 7.73, 7.25, 6.84, 5.38, 6.94, 7.48, 7.28, 6.85,
       6.14, 6.19, 6.53, 7.28, 8.31, 5.42, 5.94, 7.15, 7.36, 8.1 , 6.96,
       6.35, 7.34, 6.87, 5.99, 5.9 , 8.62, 7.43, 9.38, 6.89, 5.95, 7.66,
       5.09, 7.87, 6.07, 5.84, 8.63, 8.87, 9.58, 9.26, 8.37, 6.47, 6.86,
       8.2 , 5.84, 6.6 , 6.92, 7.56, 5.61, 5.48, 6.34, 9.16, 7.36, 7.6 ,
       5.11, 6.51, 7.56, 7.3 , 5.79, 7.47, 7.78, 8.44, 6.85, 6.97, 6.94,
       8.99, 6.59, 7.18, 7.63, 6.1 , 5.58, 8.44, 4.26, 4.79, 7.61, 8.09,
       4.73, 6.42, 7.11, 6.22, 7.9 , 6.79, 5.83, 6.63, 7.11, 5.98, 7.69,
       6.61, 7.95, 6.71, 5.13, 7.05, 7.62, 6.66, 6.13, 6.33, 7.76, 7.77,
       8.18, 5.42, 8.58, 6.94, 5.84, 8.35, 9.04, 7.12, 7.4 , 7.39, 5.23,
       6.5 , 5.12, 5.1 , 6.06, 7.33, 5.91, 6.78, 7.93, 7.29, 6.68, 6.37,
       5.84, 6.05, 7.2 , 6.1 , 5.64, 7.14, 7.91, 7.19, 7.91, 6.76, 6.93,
       4.85, 6.17, 5.84, 6.07, 5.66, 7.57, 8.28, 6.

---
*Inspecting `x` — the full array of 200 CGPA values, ranging roughly from 4.26 to 9.58.*

---

In [79]:
y

array([3.26, 1.98, 3.25, 3.67, 3.57, 2.99, 2.6 , 2.48, 2.31, 3.51, 1.86,
       2.6 , 3.65, 2.89, 3.42, 3.23, 2.35, 2.09, 2.98, 2.83, 3.16, 2.93,
       2.3 , 2.48, 2.71, 3.65, 3.42, 2.16, 2.24, 3.49, 3.26, 3.89, 3.08,
       2.73, 3.42, 2.87, 2.84, 2.43, 4.36, 3.33, 4.02, 2.7 , 2.54, 2.76,
       1.86, 3.58, 2.26, 3.26, 4.09, 4.62, 4.43, 3.79, 4.11, 2.61, 3.09,
       3.39, 2.74, 1.94, 3.09, 3.31, 2.19, 1.61, 2.09, 4.25, 2.92, 3.81,
       1.63, 2.89, 2.99, 2.94, 2.35, 3.34, 3.62, 4.03, 3.44, 3.28, 3.15,
       4.6 , 2.21, 3.  , 3.44, 2.2 , 2.17, 3.49, 1.53, 1.48, 2.77, 3.55,
       1.48, 2.72, 2.66, 2.14, 4.  , 3.08, 2.42, 2.79, 2.61, 2.84, 3.83,
       3.24, 4.14, 3.52, 1.37, 3.  , 3.74, 2.82, 2.19, 2.59, 3.54, 4.06,
       3.76, 2.25, 4.1 , 2.37, 1.87, 4.21, 3.33, 2.99, 2.88, 2.65, 1.73,
       3.02, 2.01, 2.3 , 2.31, 3.16, 2.6 , 3.11, 3.34, 3.12, 2.49, 2.01,
       2.48, 2.58, 2.83, 2.6 , 2.1 , 3.13, 3.89, 2.4 , 3.15, 3.18, 3.04,
       1.54, 2.42, 2.18, 2.46, 2.21, 3.4 , 3.67, 2.

---
*Inspecting `y` — the full array of 200 salary packages in LPA, ranging roughly from 1.37 to 4.62.*

---

In [80]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

---

### Explanation: Train-Test Split

```python
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)
```

| Variable | Contents | Size |
|---|---|---|
| `x_train` | CGPA values the model **learns from** | 160 samples (80%) |
| `x_test` | CGPA values the model is **evaluated on** | 40 samples (20%) |
| `y_train` | Salary values the model **learns from** | 160 samples (80%) |
| `y_test` | Salary values used to **check predictions** | 40 samples (20%) |

- **`test_size=0.2`** → 20% of 200 = 40 test samples, 80% = 160 training samples.
- **`random_state=2`** → Seeds the random shuffler so the **exact same split** is reproduced every run.

Note: All four variables are **1D NumPy arrays** here (shape `(160,)` or `(40,)`), which is what `MyLR` expects.

---

In [81]:
x_train.shape

(160,)

---

### Explanation: Verifying the Training Set Shape

```python
x_train.shape   # → (160,)
```

- `(160,)` confirms `x_train` is a **1D NumPy array** with 160 elements.
- The single number `160` is `x_train.shape[0]`, which is used in the `fit()` loop: `for i in range(x_train.shape[0])`.
- No second dimension means this is flat — not a matrix. This matches what `MyLR.fit()` expects.

---

In [82]:
lr = MyLR()
lr.fit(x_train, y_train)

m is  0.5579519734250721
b is  -0.8961119222429152


---

### Explanation: Training the Custom Model

```python
lr = MyLR()
lr.fit(x_train, y_train)
```

**Step 1 — `lr = MyLR()`**
Creates a fresh instance of our custom class. At this point `lr.m = None` and `lr.b = None`.

**Step 2 — `lr.fit(x_train, y_train)`**
Runs the OLS loop over all 160 training students:
- Accumulates the covariance (numerator) and variance (denominator) sums.
- Divides to get the slope $m$.
- Uses $m$ and the means to compute the intercept $b$.

**Printed output:**
```
m is  0.5579519734250721
b is  -0.8961119222429152
```

These are the **learned parameters** of the best-fit line through the 160 training students:

$$\hat{\text{Package}} = 0.5580 \cdot \text{CGPA} - 0.8961$$

> **Verification:** These values are **identical** to what `sklearn.linear_model.LinearRegression` produces on the same training split — confirming our manual OLS implementation is mathematically correct.

---

In [83]:
#now find out the m so in m we have numumerator and denominator so we will find out the numumerator and denominator separately then divide them to get m

In [84]:
lr = MyLR()
lr.fit(x_train, y_train)

m is  0.5579519734250721
b is  -0.8961119222429152


---
*The model is trained again here (same result) — confirming the output is deterministic and reproducible.*

---

In [85]:
x_test[0]

np.float64(8.58)

---

### Explanation: Inspecting the First Test Sample

```python
x_test[0]   # → 8.58
```

The first student in our test set has a CGPA of **8.58**.

We will use this value to verify the prediction in the next cell:
$$\hat{y} = 0.5580 \times 8.58 - 0.8961 = 4.7876 - 0.8961 \approx 3.89 \text{ LPA}$$

---

In [86]:
print(lr.predict(x_test[0]))

3.891116009744203


---

### Explanation: Making a Prediction

```python
print(lr.predict(x_test[0]))   # → 3.891116009744203
```

Calls `MyLR.predict()` with the first test student's CGPA (`8.58`).

Internally, `predict` computes:
$$\hat{y} = m \cdot x + b = 0.5580 \times 8.58 + (-0.8961) \approx \mathbf{3.891 \text{ LPA}}$$

**Interpretation:** The model predicts a salary package of approximately **₹3.89 LPA** for a student with CGPA 8.58.

---

## ✅ Summary — What We Built

| Component | What it does |
|---|---|
| `MyLR.__init__` | Initializes $m = \text{None}$, $b = \text{None}$ — blank slate before training |
| `MyLR.fit` | Implements OLS via a manual loop — computes covariance / variance = slope $m$, then intercept $b$ |
| `MyLR.predict` | Applies $\hat{y} = mx + b$ to any input CGPA value |
| Result: $m \approx 0.558$ | For every +1 CGPA, predicted salary increases by ~0.558 LPA |
| Result: $b \approx -0.896$ | Baseline (mathematical artifact; CGPA = 0 is outside real data range) |

**Our custom `MyLR` produces the exact same $m$ and $b$ as Scikit-Learn's `LinearRegression` — validating that both implement the same OLS math.** 🎉

---